In [2]:
import pandas as pd
import numpy as np
df = pd.read_csv('./Reviews.csv')
df

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...
...,...,...,...,...,...,...,...,...,...,...
568449,568450,B001EO7N10,A28KG5XORO54AY,Lettie D. Carter,0,0,5,1299628800,Will not do without,Great for sesame chicken..this is a good if no...
568450,568451,B003S1WTCU,A3I8AFVPEE8KI5,R. Sawyer,0,0,2,1331251200,disappointed,I'm disappointed with the flavor. The chocolat...
568451,568452,B004I613EE,A121AA1GQV751Z,"pksd ""pk_007""",2,2,5,1329782400,Perfect for our maltipoo,"These stars are small, so you can give 10-15 o..."
568452,568453,B004I613EE,A3IBEVCTXKNOH,"Kathy A. Welch ""katwel""",1,1,5,1331596800,Favorite Training and reward treat,These are the BEST treats for training and rew...


In [4]:
df.shape

(568454, 10)

In [5]:
review_df= df[['Text','Score']].sample(frac=0.5, random_state=1)

In [6]:
review_df.shape

(284227, 2)

In [7]:
review_df.head(10)

,Text,Score
288312,I love the Cherry Pie Lara bar. Best and tast...,5
431726,Melitta Cafe COllection Blanc et Noir coffee h...,5
110311,my girls absolutely loved this tuna. they were...,5
91855,The vendor is fast and dependable. The tea is ...,5
338855,UPDATE - 8/9/2010<br />A lot can happen in jus...,5
243608,I have pretty much tried them all. The dark ch...,5
152343,"Just be forewarned it is a bit ""moist."" Have ...",5
336202,Both of my mixed terrier dogs prefer this trea...,5
488611,Made this coffee with a percolator maybe that ...,1
103618,This was just the basic ingredients that i cou...,5


In [8]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

**Let's see how we can convert a document into a bag of word**

In [9]:
vectorizer = CountVectorizer()
tfidf_vectorizer = TfidfVectorizer(ngram_range=(1, 3)) # looks at n-grams of size 1 to 3, this is usefull for the context

small_example_docs = ['this is a document', 'this is another document', 'this is the third document', 'this is yet another doc', 'this is the fifth one','a document without much information information provided about the content despite the excessive length']

X_word_features = vectorizer.fit_transform(small_example_docs)
X_tfidf_features = tfidf_vectorizer.fit_transform(small_example_docs)

print(vectorizer.get_feature_names_out())
print(tfidf_vectorizer.get_feature_names_out())
print(X_word_features.toarray())
print(X_tfidf_features.toarray())

['about' 'another' 'content' 'despite' 'doc' 'document' 'excessive'
 'fifth' 'information' 'is' 'length' 'much' 'one' 'provided' 'the' 'third'
 'this' 'without' 'yet']
['about' 'about the' 'about the content' 'another' 'another doc'
 'another document' 'content' 'content despite' 'content despite the'
 'despite' 'despite the' 'despite the excessive' 'doc' 'document'
 'document without' 'document without much' 'excessive' 'excessive length'
 'fifth' 'fifth one' 'information' 'information information'
 'information information provided' 'information provided'
 'information provided about' 'is' 'is another' 'is another document'
 'is document' 'is the' 'is the fifth' 'is the third' 'is yet'
 'is yet another' 'length' 'much' 'much information'
 'much information information' 'one' 'provided' 'provided about'
 'provided about the' 'the' 'the content' 'the content despite'
 'the excessive' 'the excessive length' 'the fifth' 'the fifth one'
 'the third' 'the third document' 'third' 'third doc

**TF-IDF Definition**

TF-IDF (Term Frequency-Inverse Document Frequency) is a numerical statistic intended to reflect how important a word is to a document in a collection or corpus. It is calculated as the product of two metrics:

Term Frequency (TF)
Measures the frequency of a term $t$ within a specific document $d$:
\begin{equation}
\text{TF}(t, d) = \frac{n_{t,d}}{\sum_{k} n_{k,d}}
\end{equation}
Where $n_{t,d}$ is the number of times term $t$ appears in document $d$, and the denominator is the total number of terms in that document.

Inverse Document Frequency (IDF)
Measures how much information the word provides by looking at how common or rare it is across all documents $D$:
\begin{equation}
\text{IDF}(t, D) = \log \left( \frac{N}{|\{d \in D : t \in d\}|} \right)
\end{equation}
Where:

- $N$ is the total number of documents in the corpus.
- $|\{d \in D : t \in d\}|$ is the number of documents where the term $t$ appears.


TF-IDF Calculation
The final score is the product of these two values:
\begin{equation}
\text{TF-IDF}(t, d, D) = \text{TF}(t, d) \times \text{IDF}(t, D)
\end{equation}

**Now let us look at how we can remove stop words**

In [20]:
!pip install nltk

In [10]:
import nltk #NLTK (Natural Language Toolkit)
from nltk.corpus import stopwords

**Download the stopwords if you have not done already**

In [11]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\z72w146\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


True

In [12]:
# Set up stop words
stop_words = set(stopwords.words('english'))
stop_words

{'a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 "he's",
 'her',
 'here',
 'hers',
 'herself',
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 "i'll",
 "i'm",
 "i've",
 'if',
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'no',
 'nor',
 'not',
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'on

**Now let's vectorize with stopwords and n-gram**

In [13]:
# Vectorizers (with stopwords and n-gram range)
vectorizer = CountVectorizer(max_features=1000, stop_words=list(stop_words))
tfidf_vectorizer = TfidfVectorizer(max_features=1000, stop_words=list(stop_words), ngram_range=(1, 2))

**Transform the text data**

In [14]:
# Transform the text data
X_word_features = vectorizer.fit_transform(review_df['Text'])
X_tfidf_features = tfidf_vectorizer.fit_transform(review_df['Text'])

In [15]:
# Labels
y_text = review_df['Score']
y_text

288312    5
431726    5
110311    5
91855     5
338855    5
         ..
396926    5
230314    5
168476    5
491343    3
106153    5
Name: Score, Length: 284227, dtype: int64

In [16]:
# Train-test split using indices
num_instances = X_tfidf_features.shape[0]
X_train_indices, X_test_indices, y_train, y_test = train_test_split(
    range(num_instances), y_text, test_size=0.3, random_state=1
)

X_train = X_tfidf_features[X_train_indices, :]
X_test = X_tfidf_features[X_test_indices, :]

# X_train = X_word_features[X_train_indices, :]
# X_test = X_word_features[X_test_indices, :]

In [17]:
# Use Multinomial Naive Bayes (correct for TF-IDF features)
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)

# Make predictions
predictions = nb_model.predict(X_test)

# Evaluate accuracy
accuracy = accuracy_score(y_test, predictions)
print("Accuracy:", accuracy)

Accuracy: 0.6496616589850942
